[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-10-streaming-drift-forgetting-quantization.ipynb)

# Advanced Discussion 10 — streaming drift detectors, forgetting, and shipping a smaller model

Reproduces the chapter 10 advanced discussion: ADWIN and Page–Hinkley on a confidence stream, catastrophic forgetting versus replay, dynamic int8 quantization and ONNX export.

**Optional advanced-discussion lab** for Hands-On Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier (CPU is enough) with built-in datasets, so no download can fail.

Offline fallback: all data is simulated or bundled with scikit-learn.

In [ ]:
%pip install -q river onnx

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, time, os, io
from river import drift
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
np.random.seed(0); torch.manual_seed(0)

## 1. streaming detectors on a confidence stream that shifts at t=1000

In [ ]:
class PageHinkley:
    """Detect an upward or downward change in the mean of a stream."""
    def __init__(self, delta=0.005, lam=1.5, alpha=0.999):
        self.delta, self.lam, self.alpha = delta, lam, alpha
        self.mean = 0.0; self.n = 0; self.m_up = 0.0; self.m_dn = 0.0; self.M_up = 0.0; self.M_dn = 0.0
    def update(self, x):
        self.n += 1; self.mean += (x - self.mean) / self.n
        self.m_up = self.alpha * self.m_up + (x - self.mean - self.delta); self.M_up = min(self.M_up, self.m_up)
        self.m_dn = self.alpha * self.m_dn + (x - self.mean + self.delta); self.M_dn = max(self.M_dn, self.m_dn)
        return (self.m_up - self.M_up > self.lam) or (self.M_dn - self.m_dn > self.lam)

def stream(seed):
    r = np.random.RandomState(seed)
    return np.r_[r.normal(0.90, 0.05, 1000), r.normal(0.78, 0.07, 1000)]        # camera swap at t = 1000
res = {"ADWIN": [], "PageHinkley": []}
false_alarms = {"ADWIN": 0, "PageHinkley": 0}
for seed in range(20):
    s = stream(seed)
    for name, det in (("ADWIN", drift.ADWIN(delta=0.002)), ("PageHinkley", PageHinkley())):
        first = None
        for t, x in enumerate(s):
            fired = det.update(x) if name == "PageHinkley" else (det.update(x) or det.drift_detected)
            if fired:
                if t < 1000: false_alarms[name] += 1
                elif first is None: first = t
                if name == "PageHinkley": break
        res[name].append(None if first is None else first - 1000)
for k, v in res.items():
    got = [d for d in v if d is not None]
    print("%-12s detected %d/20 runs, median delay %s samples after the shift, false alarms before shift: %d" % (k, len(got), int(np.median(got)) if got else "-", false_alarms[k]))

## 2. catastrophic forgetting vs replay

In [ ]:
X, y = load_digits(return_X_y=True); Xn = X / 16.0
Xtr, Xte, ytr, yte = train_test_split(Xn, y, test_size=0.4, random_state=0, stratify=y)
def shift(a): return np.clip(a * 0.55 + 0.35 * (np.random.RandomState(1).rand(1, 64) > 0.5) + 0.1, 0, 1)   # 'new camera': gain + fixed offset pattern
T = lambda a: torch.tensor(a, dtype=torch.float32)
Xtr_s, Xte_s = shift(Xtr), shift(Xte)
def make(): return nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))
def fit(m, Xa, ya, epochs, lr=3e-3):
    o = torch.optim.Adam(m.parameters(), lr=lr); Xa, ya = T(Xa), torch.tensor(ya)
    for _ in range(epochs):
        p = torch.randperm(len(Xa))
        for i in range(0, len(Xa), 64):
            b = p[i:i + 64]; o.zero_grad(); F.cross_entropy(m(Xa[b]), ya[b]).backward(); o.step()
acc = lambda m, Xa, ya: (m(T(Xa)).argmax(1) == torch.tensor(ya)).float().mean().item()
base = make(); fit(base, Xtr, ytr, 60)
print("base model: old camera %.3f | new camera %.3f" % (acc(base, Xte, yte), acc(base, Xte_s, yte)))
import copy
m1 = copy.deepcopy(base); fit(m1, Xtr_s, ytr, 60)
print("fine-tuned on new camera only: old %.3f | new %.3f" % (acc(m1, Xte, yte), acc(m1, Xte_s, yte)))
m2 = copy.deepcopy(base)
keep = np.random.RandomState(0).choice(len(Xtr), int(0.2 * len(Xtr)), replace=False)
fit(m2, np.r_[Xtr_s, Xtr[keep]], np.r_[ytr, ytr[keep]], 60)
print("fine-tuned on new + 20%% replay of old: old %.3f | new %.3f" % (acc(m2, Xte, yte), acc(m2, Xte_s, yte)))

## 3. quantization + ONNX export

In [ ]:
big = nn.Sequential(nn.Linear(64, 1024), nn.ReLU(), nn.Linear(1024, 1024), nn.ReLU(), nn.Linear(1024, 10)); fit(big, Xtr, ytr, 40)
big.eval()
def size_mb(m):
    b = io.BytesIO(); torch.save(m.state_dict(), b); return b.getbuffer().nbytes / 1e6
def lat(m, n=300):
    x = T(Xte[:1]);
    with torch.no_grad():
        for _ in range(20): m(x)
        t = time.perf_counter()
        for _ in range(n): m(x)
    return (time.perf_counter() - t) / n * 1000
q = torch.ao.quantization.quantize_dynamic(big, {nn.Linear}, dtype=torch.qint8)
print("float32: %.2f MB, %.3f ms, acc %.3f" % (size_mb(big), lat(big), acc(big, Xte, yte)))
print("int8 dynamic: %.2f MB, %.3f ms, acc %.3f" % (size_mb(q), lat(q), acc(q, Xte, yte)))
try:
    torch.onnx.export(big, (T(Xte[:1]),), "m.onnx", input_names=["x"], output_names=["logits"], dynamic_axes={"x": {0: "batch"}}, dynamo=False)
    import onnx; onnx.checker.check_model(onnx.load("m.onnx")); print("onnx export ok, size %.2f MB" % (os.path.getsize("m.onnx") / 1e6))
except Exception as e:
    print("onnx export failed:", repr(e)[:200])

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.